In [ ]:
# # 1-a dalis: ECG Denoising Pipeline test with a single .npy file from DATA/ZIVE_DATA
# Uses CONFIG/pipeline_config.yaml

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Tuple
import math
import numpy as np

# --- ecg_denoising_pipeline imports (as in your original) ---
from ecg_denoising_pipeline import (
    ECGDenoisingPipeline,
    DenoisingPipelineResult,
    DenoisingPipelineConfig,
    load_denoising_config_yaml,
    check_denoising_config,
    friendly_print_denoising_cfg,
    convert_seconds_to_hms,
)

from ecg_ectopy_pipeline import resolve_model_path

# ------------- Utilities -------------

def print_heading(title: str) -> None:
    line = '-' * max(10, len(title))
    print(f'\n{title}\n{line}')

def as_seconds(
    intervals: Iterable[Tuple[int, int]], fs: float
) -> list[Tuple[float, float]]:
    """Convert sample-based intervals to seconds."""
    return [(s / fs, e / fs) for (s, e) in intervals]

def load_array_or_fail(data_dir: Path, filename: str) -> np.ndarray:
    path = (data_dir / filename).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {path}")
    arr = np.load(path, allow_pickle=False)
    if arr.size == 0:
        raise ValueError(f"ECG is empty: {path}")
    return arr


def list_npy_files(data_dir: Path) -> list[Path]:
    return sorted(data_dir.glob("*.npy"))


# ------------- Main flow -------------

def run_denoising_pipeline(
    file_name: str | None,
    data_dir: Path,
    config_path: Path,
    model_dir: Path,
    disable_motions: bool,
) -> tuple[DenoisingPipelineResult, DenoisingPipelineConfig]:
    # Paths sanity
    if not data_dir.exists():
        raise FileNotFoundError(f"Data folder not found: {data_dir}")
    if not config_path.exists():
        raise FileNotFoundError(f"Config file not found: {config_path}")
    if not model_dir.exists():
        raise FileNotFoundError(f"Model directory not found: {model_dir}")

    print_heading("Project paths")
    print("DATA_DIR:", data_dir)
    print("CONFIG  :", config_path)
    print("MODEL_DIR:", model_dir)

    # Load config
    cfg = load_denoising_config_yaml(str(config_path))
    fs = float(cfg.fs)

    print_heading("Denoising pipeline config")
    friendly_print_denoising_cfg(cfg)
    check_denoising_config(cfg)

    # Pick ECG file
    if file_name is None:
        files = list_npy_files(data_dir)
        if not files:
            raise FileNotFoundError(f"No .npy files in {data_dir}")
        file_path = files[0]
        file_name = file_path.name
        print(f"\nNo --file provided. Using first file found: {file_name}")
    else:
        file_path = (data_dir / file_name).resolve()

    # Load ECG
    print_heading("Input data")
    print(f"File: {file_name}")
    x = load_array_or_fail(data_dir, file_name)
    len_secs = math.ceil(len(x) / fs)
    h, m, s = convert_seconds_to_hms(len_secs)
    print(f"len(ecg): {len(x)} samples (~{len_secs:.1f} s) | {h:02d}:{m:02d}:{s:02d}")

    # Resolve model path (robust to ~ and absolute/relative)
    cfg.motions.model_name = resolve_model_path(model_dir, cfg.motions.model_name)

    if disable_motions:
        cfg.motions.enabled = False

    print_heading("UNet model")
    print("UNet model path:", cfg.motions.model_name)
    print("Motions enabled:", bool(getattr(cfg.motions, "enabled", True)))

    # Run pipeline
    pipe = ECGDenoisingPipeline(cfg)
    res = pipe.run(x, gaps_indices=[])

    return res, cfg


# start0
"""
Resolve key folders:
    - HERE: the project subfolder that contains CONFIG and sibling DATA/MODEL_UNET
    - ROOT: project root (parent of HERE)
    """
HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
# HERE  0_SELECT_ZIVE_DATA_2023
ROOT = HERE.parents[0]    # PROJECT_TRAIN_UNET
print("HERE:", HERE)
print("ROOT:", ROOT)

data_dir = ROOT / "DATA_ORIG" / "ecg_zive_npy"
cfg_dir = ROOT / "CONFIG" / "denoising_config.yaml"
model_dir = ROOT / "MODEL_UNET"  # PROJECT_TRAIN_UNET/MODEL_UNET

print("DATA_DIR:", data_dir)
print("CFG_DIR :", cfg_dir)


In [ ]:
print("MODEL_DIR:", model_dir)

   # "Run ECG denoising pipeline on a single .npy ECG file.\n"
   # "If --file is not provided, the first .npy in DATA_DIR is used."

file_name = "1005_10.npy"
file_name = "1065_11.npy"

# Execute the pipeline in the notebook with explicit arguments
res_denoising, cfg_denoising = run_denoising_pipeline(
file_name=file_name,
data_dir=data_dir,
config_path=cfg_dir,
model_dir=model_dir,
disable_motions=False,
)

# Results
print_heading("Results")
print(f"len_original: {len(res_denoising.ecg_orig)}")
print(f"len_start   : {len(res_denoising.ecg_start)}")
print(f"len_denoised   : {len(res_denoising.ecg_denoised)}")

print("\nMaps (sample intervals):")
print("map_gaps    :", res_denoising.map_gaps)
print("map_outliers:", res_denoising.map_outliers)
print("map_rdropouts:", res_denoising.map_rdropouts)
print("map_motions :", res_denoising.map_motions)

print_heading("Detected intervals (in samples)")
print("Outliers (start):", res_denoising.outliers_indices_start)
print("Rdropouts (nout):", res_denoising.rdropouts_indices_nout)
print("Motions (nrd)   :", res_denoising.motions_indices_nrd)

print_heading("Detected intervals (in seconds)")
print("Outliers (start):", as_seconds(res_denoising.outliers_indices_start, cfg_denoising.fs))
print("Rdropouts (nout):", as_seconds(res_denoising.rdropouts_indices_nout, cfg_denoising.fs))
print("Motions (nrd)   :", as_seconds(res_denoising.motions_indices_nrd, cfg_denoising.fs))



In [ ]:
# 2-a dalis: ECTOPY: TEST premature beats detection, naudojant išvalytą nuo triukšmų signalą
# Naudoja configuracijos failą config_ectopy.yaml

from ecg_denoising_pipeline import DenoisingPipelineResult
from ecg_ectopy_pipeline import (
    EctopyPipelineConfig,
    ECGEctopyPipeline,
    EctopyPipelineResult,
    run_ectopy_pipeline,
)

# start1
"""
Resolve key folders:
    - HERE: the project subfolder that contains CONFIG and sibling DATA/MODEL_VU_CNN
    - ROOT: project root (parent of HERE)
    """
HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
ROOT = HERE.parents[0]    # PROJECT_TRAIN_UNET
print("ROOT:", ROOT)

# Configuration and model directories
cfg_dir = ROOT / "CONFIG" / "ectopy_config.yaml"
model_dir = ROOT / "MODEL_VU_CNN"  # PROJECT_TRAIN_UNET/MODEL_VU_CNN

print("CFG_DIR :", cfg_dir)
print("MODEL_DIR:", model_dir)


# "Run ECG ectopy pipeline on a signal from a ecg_denoising_pipeline
# Execute the pipeline in the notebook with explicit arguments
res_ectopy = run_ectopy_pipeline(
    file_name=file_name,
    fs=int(cfg_denoising.fs),
    res_denoising = res_denoising,
    ectopy_config_path=cfg_dir,
    ectopy_model_dir=model_dir,
    disable_ectopy_removing=False,
)

# Results
print_heading("ECTOPY pipeline results:")
print("rpeaks_on_denoised_df head (first 100 rows):")
print(res_ectopy.rpeaks_on_denoised_df.head(100))


In [ ]:
# Vaizdavimas su annotacijomis ir predikcijomis

# Vaizduojame signalą su ectopijomis : res.ecg_start
# naudojame rpeaks_on_start_df (pred_df) ir annot_df
# Show denoised signal with ectopy //////////////////////////////////

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional
import numpy as np
import pandas as pd

import numpy.typing as npt

# from ecg_ectopy_pipeline import load_nonzero_annots
# from ecg_ectopy_pipeline.ectopy_plotting import (
#     plot_ecg_annot_pred,
#     divide_signal_into_fragments,
#     plot_gap_legend,
# )

from ecg_denoising_pipeline import (
    load_nonzero_annots,
    plot_ecg_annot_pred,
    divide_signal_into_fragments,
    plot_gap_legend,
)


from ecg_denoising_pipeline import map_mark_denoised_to_start
# from step_beats_no_ectopy import map_rpeaks_denoised_to_start

IntArray1D = npt.NDArray[np.int_]
BoolArray1D = npt.NDArray[np.bool_]

fs = cfg_denoising.fs

flag_show_denoised_with_ectopy = True
# Show denoised signal with ectopy //////////////////////////////////

if flag_show_denoised_with_ectopy:
    
    print("\nShow final signal with ectopy annotations and predictions:")
    
    # Prepare data for plotting

    # 1. Iš išvalyto nuo triukšmų signalo randame R-peak'us rpeaks_on_denoised
    #  ir remapiname į R-peak'us į pradinį signalą
    rpeaks_on_denoised = res_ectopy.rpeaks_on_denoised_df["rpeak"].astype(int).tolist()
    rpeaks_on_start, rpeaks_on_gap = map_mark_denoised_to_start( rpeaks_on_denoised, res_denoising )
    rpeaks_on_start = np.array(rpeaks_on_start, dtype=int)

    # 2. Jei yra ectopy annotacijos,jas užkrauname
    annot_df_: Optional[pd.DataFrame] = load_nonzero_annots(data_dir, file_name)
    if annot_df_ is None:
        print("\nNo non-zero annotations found.")
    else:
        print(f"\nlen(annot_df_): {len(annot_df_)}")

    # 3. Remapiname rpeaks_on_denoised_df, gautą iš get_beats_ml_classes į start signalą
    # Build dataframe in 'start' coordinates (keeps Index and pred)
    rpeaks_on_start_df = res_ectopy.rpeaks_on_denoised_df.copy()
    rpeaks_on_start_df["rpeak"] = rpeaks_on_start

    print(f"Mapped {len(rpeaks_on_start_df)} R-peaks from final -> start")
    pred_df_ = rpeaks_on_start_df.loc[rpeaks_on_start_df["pred"].ne(0)].copy()
    if pred_df_.empty:
        pred_df_ = None

    PORTION_LENGTH_SECS = 20
    DISPLAY_AS_SECONDS = True
    SAVE_PLOTS = False
    PLOT_DIR = HERE / "PLOTS" / "ectopy_plots"
    PLOT_SUFFIX = "with_ectopy"
    fileName = "ecg_start"

    ecg_signal = res_denoising.ecg_start

    rpeaks_on_start_secs = np.array([x/fs for x in rpeaks_on_start])
    print(f"rpeaks_start (len {len(rpeaks_on_start)}): {len(rpeaks_on_start_secs)}")

    # annot_df_ = annot_df.loc[annot_df["annot"] != 0].copy()
    # pred_df = rpeaks_on_start_df
    # pred_df_ = pred_df.loc[pred_df["pred"] != 0].copy()

    # Convert outliers indices to seconds for reporting
    outliers_idx_seconds = [(s / fs, e / cfg_denoising.fs) for (s, e) in res_denoising.projected_to_start['outliers']]
    print(f"outliers_idx_seconds: {outliers_idx_seconds}")
    rdropouts_idx_seconds = [(s / fs, e / cfg_denoising.fs) for (s, e) in res_denoising.projected_to_start['rdropouts']]
    print(f"rdropouts_idx_seconds: {rdropouts_idx_seconds}")
    motions_idx_seconds = [(s / fs, e / cfg_denoising.fs) for (s, e) in res_denoising.projected_to_start['motions']]
    print(f"motions_idx_seconds: {motions_idx_seconds}")

    fragment_samples = divide_signal_into_fragments(ecg_signal, int(PORTION_LENGTH_SECS * fs))
    fragment_secs = [(start / fs, end / fs) for start, end in fragment_samples]
    print("Fragments (s):", [(round(s, 1), round(e, 1)) for s, e in fragment_secs])
 
    # Plot small rectangles and notes in one line
    plot_gap_legend('outliers', 'rdropouts', 'motions')
    print("\noutliers-> yellow, rdropouts-> blue, motions-> red")

    # gap_layers = [
    #     GapLayer(tuple(gap1_indices_secs or []), "red", 0.5),
    #     GapLayer(tuple(gap2_indices_secs or []), "blue", 0.5),
    #     GapLayer(tuple(gap3_indices_secs or []), "yellow", 0.7),
    # ]

    # plot_ecg_annot_pred(file_name, ecg_signal, fs, show_frag_start_secs, show_frag_end_secs,
    #         portion_to_plot_secs, plot_save_dir, 'debug_plots', None,  ectopy_indices_secs,  None,  None,
    #         None, None,  0, rpeaks_with_ectopy_secs, True)
 
 
    next_fragment_idx = 1
    for start_sec, end_sec in fragment_secs:
        next_fragment_idx = plot_ecg_annot_pred(
            fileName=fileName,
            ecg_signal=ecg_signal,
            fs=fs,
            plot_signal_from_in_secs=start_sec,
            plot_signal_to_in_secs=end_sec,
            portion_length_in_secs=PORTION_LENGTH_SECS,
            plot_save_dir=str(PLOT_DIR) if SAVE_PLOTS else None,
            save_mark=PLOT_SUFFIX,
            recID=None,
            gap1_indices_secs=motions_idx_seconds, # red
            gap2_indices_secs=rdropouts_idx_seconds, # blue
            gap3_indices_secs=outliers_idx_seconds, # yellow
            rpeak_indices_secs=rpeaks_on_start_secs,
            annot_df=annot_df_,
            pred_df=pred_df_,
            tol_samples=10,
            flag_secs=DISPLAY_AS_SECONDS,
        )

    print("Finished plotting", file_name)

In [ ]:
# Įvertiname ectopijų aptikimo tikslumą naudojant annotacijas ir predikcijas


# Klaidos skaičiavimas
# dalis paimta iš TEST_VU_CNN/zive_aritmijos_klasifikacija_NN_algoritmas_one_keras3.ipynb

from ecg_ectopy_pipeline import (
    merge_rpeaks_with_annotations,
    read_df_annot,
    print_classification_results,
    evaluate_binary_classification,
)

# from step_beats_no_ectopy import map_rpeaks_denoised_to_start


# +++++++  RPIKŲ IR EKSTRASYSTOLIŲ INDEKSŲ PERSKAIČIAVIMAS Į PRADINĮ SIGNALĄ +++++++++++++++++++

# rpikų indeksų su klasių numeriais perskaičiavimas į pradinių duomenų signalą ecg_start

# rezultatas: rpeaks_on_start_df

# pirmiausiai perskaičiuojame rpeaks_on_denoised į rpeaks_on_start
# naudojame žemėlapius iš denoising pipeline: res.map_motions, res.map_rdropouts, res.map_outliers, res.map_gaps

# nuskaitome anotoutacijas iš medikų  ir sulyginame su ML klasifikacija

# 1. Iš išvalyto nuo triukšmų signalo randame R-peak'us rpeaks_on_denoised
#  ir remapiname į R-peak'us į pradinį signalą
rpeaks_on_denoised = res_ectopy.rpeaks_on_denoised_df["rpeak"].astype(int).tolist()
rpeaks_on_start, rpeaks_on_gap = map_mark_denoised_to_start( rpeaks_on_denoised, res_denoising )
rpeaks_on_start = np.array(rpeaks_on_start, dtype=int)

# Build dataframe in 'start' coordinates (keeps Index and pred)
rpeaks_on_start_df = res_ectopy.rpeaks_on_denoised_df.copy()
rpeaks_on_start_df["rpeak"] = rpeaks_on_start
# Backward-compatible alias if used later in the notebook
rpeaks_df_start = rpeaks_on_start_df

print(f"Mapped {len(rpeaks_on_start_df)} R-peaks from final -> start")
# print("\nrpeaks_on_start_df:")
# print(rpeaks_on_start_df.head(10))

            # ++++++++++++++++++++++++++  ECG PŪPSNIŲ KLASIFIKACIJOS SULYGINIMAS SU MEDIKŲ ANOTACIJOMIS 
 
# Nuskaitome paciento įrašo medikų anotacijas atr_symbol_orig ('N', 'S', 'V', 'U')
# ir jų indeksus atr_sample_orig (rpeaks vietas signal masyve)
print(f"\nfileName: {file_name}")
annot_df = read_df_annot(data_dir, file_name)
print(f"\nlen(annot_df): {len(annot_df)}")
# print("\nannot_df:")
# print(annot_df.head(20))

tolerance = 20
print("\ntolerance:", tolerance)

    # Merge the dataframes
df_matched, df_unmatched, not_found_count, description = merge_rpeaks_with_annotations(rpeaks_on_start_df,
                                                                    annot_df, tolerance=tolerance)

print(description)
print(f"\nlen(df_matched): {len(df_matched)}")
df_matched = df_matched.sort_values(by='diff', ascending=False)
# print(df_matched.head(10))
print(f"\nlen(df_unmatched): {len(df_unmatched)}")
df_unmatched["rpeak_sec"] = df_unmatched["rpeak"] / fs
df_unmatched["closest_rpeak_annot_sec"] = df_unmatched["closest_rpeak_annot"] / fs
# print(df_unmatched.head(20))

# Remove annot == 3 from df_matched
removed_count = (df_matched['annot'] == 3).sum()
df_matched = df_matched[df_matched['annot'] != 3].reset_index(drop=True)
print(f"\nRemoved {removed_count} rows with annot == 3")

# (unique_labels_annot, counts_annot) = np.unique(df_matched['annot'].values, return_counts=True)
counts = df_matched['annot'].value_counts(dropna=False)
unique_labels_annot = counts.index.to_numpy()
counts_annot = counts.to_numpy()
print("\nLabels for annot: ", unique_labels_annot, counts_annot, "Total:", counts_annot.sum())

# (unique_labels_pred, counts_pred) = np.unique(df_matched['pred'].values, return_counts=True)
counts = df_matched['pred'].value_counts(dropna=False)
unique_labels_pred = counts.index.to_numpy()
counts_pred = counts.to_numpy()
print("Labels for pred: ", unique_labels_pred, counts_pred, "Total:", counts_pred.sum())


# Surandame klasifikavimo tikslumą ir išvedame rezultatus
print("\nKlasifikavimo tikslumas")
comment = []
test_labels = df_matched['annot'].values
pred_labels = df_matched['pred'].values
print_classification_results(test_labels, pred_labels, comment)

# Sulyginimui su binarine klasifikacija perskaičiuojame labels: N=0, S=1, V=1, U=1
# Ensure ndarray to avoid cases where a scalar bool leaks in and lacks `.astype`
test_labels_bin = (np.asarray(test_labels) != 0).astype(int)
pred_labels_bin = (np.asarray(pred_labels) != 0).astype(int)
print("\nClassification results for binary case")
evaluate_binary_classification(test_labels_bin, pred_labels_bin, positive_class=1)



In [ ]:
# Vaizduojame švarų signalą be ectopijų //////////////////////////////////

from ecg_denoising_pipeline import get_rpeaks

flag_show_denoised_without_ectopy = True

# Švarus signalas be ectopijų: res.ecg_denoised_no_ectopy

if flag_show_denoised_without_ectopy:
    
    print("\nShow denoised signal without ectopy annotations and predictions:")
    
    # Prepare data for plotting

    # 1. Iš išvalyto nuo triukšmų ir ekstrasistoliųs signalo randame R-peak'us rpeaks_on_no_ectopy
    ecg_signal = res_ectopy.ecg_no_ectopy
    # rpeaks_on_no_ectopy = map_rpeaks_denoised_to_start(
    rpeaks_on_no_ectopy, rpeaks_on_no_ectopy_secs = get_rpeaks(ecg_signal, fs=fs)


    PORTION_LENGTH_SECS = 20
    DISPLAY_AS_SECONDS = True
    SAVE_PLOTS = False
    PLOT_DIR = HERE / "PLOTS" / "ectopy_plots"
    PLOT_SUFFIX = "without_ectopy"
    fileName = "ecg_denoised_no_ectopy"

    print(f"rpeaks_start (len {len(rpeaks_on_no_ectopy)}): {len(rpeaks_on_no_ectopy_secs)}")

    fragment_samples = divide_signal_into_fragments(ecg_signal, int(PORTION_LENGTH_SECS * fs))
    fragment_secs = [(start / fs, end / fs) for start, end in fragment_samples]
    print("Fragments (s):", [(round(s, 1), round(e, 1)) for s, e in fragment_secs])
 
    next_fragment_idx = 1
    for start_sec, end_sec in fragment_secs:
        next_fragment_idx = plot_ecg_annot_pred(
            fileName=fileName,
            ecg_signal=ecg_signal,
            fs=fs,
            num_fragment=next_fragment_idx,
            plot_signal_from_in_secs=start_sec,
            plot_signal_to_in_secs=end_sec,
            portion_length_in_secs=PORTION_LENGTH_SECS,
            plot_save_dir=str(PLOT_DIR) if SAVE_PLOTS else None,
            save_mark=PLOT_SUFFIX,
            recID=None,
            rpeak_indices_secs=rpeaks_on_no_ectopy_secs,
            tol_samples=0,
            flag_secs=DISPLAY_AS_SECONDS,
        )

    print("Finished plotting", fileName)
